In [1]:
# torch(딥러닝 계산 라이브러리)를 가져온다 — GPU 확인에 쓰려고
import torch
# transformers(허깅페이스 모델 라이브러리)를 가져온다 — 버전 확인에 쓰려고
import transformers
# sys(파이썬 실행 정보 모듈)를 가져온다 — 지금 어떤 파이썬이 이 노트북을 돌리는지 확인하려고
import sys
# os(운영체제와 대화하는 모듈)를 가져온다 — 노트북의 현재 작업 폴더를 확인하려고
import os

# 이 노트북을 실행하는 파이썬 파일 경로를 출력한다 — transcript-pii-masker\.venv 가 들어 있어야 커널이 제대로 잡힌 것
print("파이썬 위치 :", sys.executable)
# 노트북의 현재 작업 폴더를 출력한다 — transcript-pii-masker 로 끝나야 이후 'samples/...' 같은 상대 경로가 맞게 동작한다
print("작업 폴더   :", os.getcwd())
# transformers 버전을 출력한다 — 5.19.0 이어야 강의안과 같다
print("transformers :", transformers.__version__)
# GPU 사용 가능 여부를 출력한다 — True면 GPU, False면 CPU로 돌게 된다
print("GPU 사용 가능 :", torch.cuda.is_available())

C:\Users\winh\transcript-pii-masker\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


파이썬 위치 : C:\Users\winh\transcript-pii-masker\.venv\Scripts\python.exe
작업 폴더   : C:\Users\winh\transcript-pii-masker
transformers : 5.19.0
GPU 사용 가능 : True


In [2]:
# re(정규식 모듈)를 가져온다 — '숫자숫자:숫자숫자' 같은 글자 패턴으로 타임스탬프 줄을 알아보려고
import re
# Path(파일 경로를 다루는 도구)를 가져온다 — 윈도우의 \ 와 / 차이를 신경 쓰지 않고 경로를 만들 수 있어서
from pathlib import Path

# 샘플 파일 경로를 만든다
# 노트북이 프로젝트 맨 위(transcript-pii-masker)에 있으므로, 바로 아래 samples 폴더를 가리키면 된다
SAMPLE = Path("samples") / "sample_transcript.txt"

# 타임스탬프 패턴을 미리 만들어 둔다 (re.compile = 패턴을 한 번 해석해 두고 여러 번 재사용하려고)
#   ^             : 줄의 시작
#   \d{1,2}:\d{2} : 숫자 1~2개, 콜론, 숫자 2개 (예: 00:07, 3:14)
#   (:\d{2})?     : 콜론+숫자 2개가 '있을 수도 없을 수도' — 1시간 넘는 녹음의 01:02:15 형태까지 받으려고
#   $             : 줄의 끝
#   ^와 $로 양 끝을 묶었기 때문에, 줄 '전체'가 이 모양일 때만 타임스탬프로 인정한다
#   (발언 중간에 '2:30에 만나요'처럼 시각이 나와도 그 줄은 타임스탬프로 오인되지 않는다)
TIMESTAMP = re.compile(r"^\d{1,2}:\d{2}(:\d{2})?$")

# 파일을 UTF-8 한글로 읽어서 줄 단위 리스트로 쪼갠다 — 갤럭시 파일이 UTF-8 형식이라서
raw_lines = SAMPLE.read_text(encoding="utf-8").splitlines()

# (타임스탬프, 발언) 짝을 담을 빈 리스트를 만든다 — 결과를 볼 때 '몇 분 몇 초 발언인지' 함께 보려고
utterances = []
# 바로 앞에서 본 타임스탬프를 기억할 변수 — 처음엔 아직 본 게 없으니 빈 글자
current_ts = ""

# 파일의 줄을 하나씩 꺼내 반복한다
for line in raw_lines:
    # 줄 앞뒤 공백을 제거한다 — 갤럭시 파일은 발언 줄 끝에 공백이 하나 붙어 있어서
    s = line.strip()
    # 빈 줄이면 아무것도 하지 않고 다음 줄로 넘어간다
    if not s:
        continue
    # 타임스탬프 줄이면 그 값을 기억해 두고 다음 줄로 넘어간다 (모델에는 넣지 않는다)
    if TIMESTAMP.match(s):
        current_ts = s
        continue
    # 여기까지 왔다면 발언 줄이다 — 직전에 기억한 타임스탬프와 짝지어 리스트에 넣는다
    utterances.append((current_ts, s))

# 발언 줄이 몇 개인지 출력한다 — 샘플 파일 기준 33개가 나와야 정상
print("발언 줄 수 :", len(utterances))
# 첫 번째 짝을 출력해 형태를 확인한다
print(utterances[0])
# 마지막 짝을 출력한다 — 파일 끝까지 빠짐없이 읽혔는지 확인하려고
print(utterances[-1])

발언 줄 수 : 33
('00:00', '네 안녕하세요. 오늘 상담 맡게 된 진로상담사 박지영입니다. 편하게 앉으세요.')
('03:35', '네 감사합니다. 서연이도 오늘 수고했어요.')


In [3]:
# transformers에서 pipeline 함수를 가져온다 — 토크나이저(글자→숫자)+모델+후처리를 한 번에 묶어 주는 도구
from transformers import pipeline
# torch를 가져온다 — GPU 사용 가능 여부를 묻기 위해
import torch

# GPU가 있으면 0번 GPU를, 없으면 -1(CPU)을 쓰도록 정한다 — pipeline은 장치를 번호로 받는다
device = 0 if torch.cuda.is_available() else -1

# 비교할 두 모델 이름을 리스트로 둔다 — 같은 녹취록으로 나란히 비교하기 위해
model_names = [
    "FrameByFrame/korean-pii-e5-base",               # 1순위: 대화체 한국어로 학습, MIT 라이선스
    "vitus9988/klue-roberta-small-ner-identified",   # 2순위: pipeline 사용이 간단, 라이선스 없음
]

# 모델 이름을 하나씩 꺼내 반복한다
for name in model_names:
    # 어느 모델의 결과인지 눈에 띄게 구분선과 함께 출력한다
    print("\n" + "=" * 25, name, "=" * 25)
    # try로 감싼다 — 1순위 모델이 v5에서 에러가 나도 셀이 멈추지 않고 2순위 시험을 계속하게 하려고
    try:
        # 토큰 분류 pipeline을 만든다
        ner = pipeline(
            "token-classification",          # 태스크: 문장의 토큰(단어 조각)마다 라벨을 붙이는 분류
            model=name,                      # 이번 반복에서 시험할 모델 (처음이면 인터넷에서 내려받는다)
            device=device,                   # 위에서 정한 장치(GPU 또는 CPU)
            aggregation_strategy="simple",   # 쪼개진 토큰 라벨을 '김서연'처럼 하나의 덩어리로 묶어 준다
        )
        # 이 모델이 찾아낸 덩어리 총개수를 셀 변수 — 처음엔 0
        total = 0
        # 셀 2에서 만든 (타임스탬프, 발언) 짝을 하나씩 꺼내 반복한다
        for ts, text in utterances:
            # 발언 한 줄을 모델에 넣어 개인정보 덩어리 목록을 받는다
            # 결과는 [{'entity_group': 라벨, 'score': 확신도, 'word': 찾은 글자, 'start': 시작위치, 'end': 끝위치}, ...]
            spans = ner(text)
            # 이 줄에서 아무것도 못 찾았으면 출력하지 않고 다음 줄로 넘어간다 — 화면이 너무 길어지지 않게
            if not spans:
                continue
            # 찾은 개수를 총개수에 더한다
            total += len(spans)
            # 몇 분 몇 초 발언인지와 원문을 출력한다
            print(f"\n[{ts}] {text}")
            # 찾아낸 덩어리를 하나씩 출력한다
            for sp in spans:
                # 원문에서 start~end 위치를 직접 잘라 온다
                # 모델의 'word' 대신 원문을 잘라 쓰는 이유: 'word'는 토크나이저가 띄어쓰기를 바꿔 놓을 수 있어서
                #   (예: '010 - 3847 - 2915'). 나중에 실제로 가릴 때 쓰는 것도 이 위치값이므로, 위치가 정확한지 여기서 확인한다
                original = text[sp["start"]:sp["end"]]
                # 라벨, 원문 조각, 위치, 확신도를 보기 좋게 한 줄로 출력한다
                print(f"   {sp['entity_group']:<14} {original!r:<35} {sp['start']}-{sp['end']}  {sp['score']:.2f}")
        # 이 모델이 찾은 총개수를 출력한다
        print(f"\n▶ {name} 탐지 총 {total}개")
    # 에러가 나면 셀을 멈추지 않고 에러 내용만 출력한다
    except Exception as e:
        # 에러 종류와 메시지를 출력한다 — 1순위를 포기할지 판단하는 근거가 된다
        print("▶ 실패 :", type(e).__name__, "-", e)


========================= FrameByFrame/korean-pii-e5-base =========================


C:\Users\winh\transcript-pii-masker\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\winh\.cache\huggingface\hub\models--FrameByFrame--korean-pii-e5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\winh\transcript-pii-masker\.venv\Lib\site-packages\huggingface_hub\file_download.py:149


[00:00] 네 안녕하세요. 오늘 상담 맡게 된 진로상담사 박지영입니다. 편하게 앉으세요.
   private_person '박지'                                26-28  1.00
   private_person '영'                                 28-29  1.00

[00:07] 안녕하세요. 저는 서연이 아빠 이정훈이고요 얘가 김서연이에요. 아 성이 달라서 헷갈리실 수 있는데 재혼 가정이라서요.
   private_person '서'                                 10-11  1.00
   private_person '연'                                 11-12  1.00
   private_person '이정'                                17-19  1.00
   private_person '훈'                                 19-20  1.00
   private_person '김서'                                27-29  1.00
   private_person '연'                                 29-30  1.00

[00:21] 네 괜찮습니다. 서연 학생은 지금 몇 학년이에요?
   private_person '서'                                 9-10  1.00
   private_person '연'                                 10-11  1.00

[00:38] 2009년 3월 15일이요.
   private_date   '2009년 3월 15'                       0-11  1.00
   private_date   '일'                                 11-12  1.00

[00:48] 네 제 번호로 해 주

C:\Users\winh\transcript-pii-masker\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\winh\.cache\huggingface\hub\models--vitus9988--klue-roberta-small-ner-identified. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1352.01it/s]



[00:00] 네 안녕하세요. 오늘 상담 맡게 된 진로상담사 박지영입니다. 편하게 앉으세요.
   PS             '박지영'                               26-29  0.97

[00:07] 안녕하세요. 저는 서연이 아빠 이정훈이고요 얘가 김서연이에요. 아 성이 달라서 헷갈리실 수 있는데 재혼 가정이라서요.
   PS             '서연'                                10-12  0.81
   PS             '이정훈'                               17-20  0.97
   PS             '김서연'                               27-30  0.98

[00:38] 2009년 3월 15일이요.
   DT             '월 15일'                             7-12  0.68

[00:48] 네 제 번호로 해 주세요. 010-3847-2915입니다.
   PH             '010-3847-2915'                     15-28  0.99

[00:59] 공일공 칠이육삼 사팔일오요.
   PS             '공일'                                0-2  0.91

[01:10] seoyeon.kim@example.com으로 보내 주세요. 아니면 제 거 jh.lee77@example.co.kr도 괜찮아요.
   EM             'seoyeon.kim@example.com'           0-23  0.98
   EM             'jh.lee77@example.co.kr'            42-64  0.98

[01:31] 그리고 학교에서 공문 처리하려면 주민번호가 필요하다고 해서요. 090315-4123456 맞나요?
   RN             '090315-4123456'           

In [4]:
%%writefile app/schemas.py
# 위 줄(%%writefile)은 이 셀의 내용을 실행하지 않고 app/schemas.py 라는 파일로 저장하라는 주피터 명령이다

# Field(필드의 규칙을 적는 도구), BaseModel(데이터 설계도의 부모 클래스), field_validator(직접 검사 함수 표시)를 가져온다
from pydantic import BaseModel, Field, field_validator
# Literal(정해진 값 몇 개만 허용하는 타입)을 가져온다 — 출처를 "regex" 또는 "model" 둘 중 하나로만 제한하려고
from typing import Literal


# ── 입력 설계도 ──────────────────────────────
# BaseModel을 상속한 클래스 = "이런 모양의 JSON만 받는다"는 설계도
class MaskRequest(BaseModel):
    # 마스킹할 녹취록 텍스트. ... = 기본값 없음 = 필수
    # min_length=1     : 한 글자도 없는 입력을 막는다
    # max_length=50_000: 29분 녹취록이 약 1만 자 → 2시간 반 분량까지 허용 (_는 읽기 쉽게 넣은 자릿수 구분, 50000과 같음)
    text: str = Field(..., min_length=1, max_length=50_000,
                      description="갤럭시 녹음 앱에서 내보낸 녹취록 텍스트")
    # 모델 확신도 기준. 선택 항목이라 기본값 0.5를 준다
    # ge=0.0 : greater or equal = 0.0 '이상' (0.0 자체도 허용)
    # le=1.0 : less or equal    = 1.0 '이하' (1.0 자체도 허용) — 확신도는 0~1 사이 값이므로
    threshold: float = Field(0.5, ge=0.0, le=1.0,
                             description="이 값 이상인 모델 탐지만 가린다")
    # 가명 처리 여부. 선택 항목이라 기본값 True
    pseudonymize: bool = Field(True, description="True면 [이름_A]처럼 같은 사람에게 같은 가명")

    # text 필드를 직접 검사하는 함수를 붙인다 — Field로는 '공백만 있는 입력'을 못 막아서
    @field_validator("text")
    # classmethod = 개별 객체가 아니라 설계도(클래스)에 붙는 함수라는 표시 (field_validator가 요구하는 형식)
    @classmethod
    # v = 사용자가 보낸 text 값
    def not_blank(cls, v: str) -> str:
        # v.strip() = 앞뒤 공백·줄바꿈을 지운 결과. 그게 빈 글자("")면 거짓이므로 not이 붙어 참이 된다
        if not v.strip():
            # ValueError를 던지면 Pydantic이 422 응답으로 바꿔 준다
            raise ValueError("공백만 있는 텍스트는 처리할 수 없습니다")
        # 통과하면 원래 값을 그대로 돌려준다 (공백을 지운 값이 아니라 원본 — 녹취록 형식을 보존하려고)
        return v


# ── 출력 설계도 ──────────────────────────────
# 탐지 한 건의 모양. 원래 개인정보 글자는 일부러 넣지 않는다 (응답 전체에 개인정보가 없게)
class Entity(BaseModel):
    # 유형 (예: "이름", "전화번호")
    type: str
    # 그 발언의 타임스탬프 (예: "00:48") — 사용자가 원본에서 위치를 찾아 검토할 수 있게
    timestamp: str
    # 누가 찾았는지 — 정규식이면 "regex", 모델이면 "model"만 허용
    source: Literal["regex", "model"]


# 응답 전체의 모양
class MaskResponse(BaseModel):
    # 성공 여부
    success: bool
    # 가린 결과 전체 (타임스탬프 줄 포함, 원래 형식 그대로)
    masked_text: str
    # 유형별 개수 — dict[str, int] = "글자 열쇠 → 정수 값" 사전
    summary: dict[str, int]
    # 탐지 목록 — list[Entity] = Entity 설계도를 따르는 항목들의 목록
    entities: list[Entity]
    # 처리 시간(밀리초)
    processing_ms: float

Writing app/schemas.py


In [5]:
# 방금 만든 파일에서 입력 설계도를 가져온다
from app.schemas import MaskRequest
# Pydantic의 검증 실패 에러를 가져온다 — 일부러 틀린 입력을 넣었을 때 잡으려고
from pydantic import ValidationError

# (시험 이름, 넣어 볼 값) 짝 목록 — 정상 1개 + 막혀야 하는 4개
cases = [
    ("정상",          {"text": "00:00\n안녕하세요 "}),
    ("빈 글자",       {"text": ""}),
    ("공백만",        {"text": "     "}),
    ("너무 김",       {"text": "가" * 50_001}),
    ("threshold 2",  {"text": "안녕", "threshold": 2}),
]

# 시험을 하나씩 꺼내 반복한다
for name, data in cases:
    # try로 감싼다 — 검증 실패는 에러로 튀어나오므로
    try:
        # **data = 사전을 펼쳐 이름=값 인자로 넘긴다. 설계도 규칙을 통과하면 객체가 만들어진다
        req = MaskRequest(**data)
        # 통과하면 '통과'와 기본값이 채워진 결과를 출력한다
        print(f"{name:<12} → 통과  threshold={req.threshold}, pseudonymize={req.pseudonymize}")
    # 검증에 실패하면 여기로 온다
    except ValidationError as e:
        # 첫 번째 에러의 이유(msg)만 짧게 출력한다
        print(f"{name:<12} → 막힘  {e.errors()[0]['msg']}")

정상           → 통과  threshold=0.5, pseudonymize=True
빈 글자         → 막힘  String should have at least 1 character
공백만          → 막힘  Value error, 공백만 있는 텍스트는 처리할 수 없습니다
너무 김         → 막힘  String should have at most 50000 characters
threshold 2  → 막힘  Input should be less than or equal to 1


In [6]:
%%writefile app/detectors.py
# 위 줄(%%writefile)은 이 셀의 내용을 실행하지 않고 app/detectors.py 라는 파일로 저장하라는 주피터 명령이다
"""
규칙 기반 부품: 타임스탬프 판별, 정규식 탐지, 탐지 조각 합치기.
모델 없이 동작하므로 서버를 띄우지 않고도 단독으로 시험할 수 있다.
"""
# re(정규식 모듈)를 가져온다 — 글자 '패턴'으로 개인정보와 타임스탬프를 찾으려고
import re


# ── 1. 타임스탬프 판별 ───────────────────────
# 타임스탬프 패턴을 미리 해석해 둔다 (re.compile = 한 번 해석해 두고 여러 줄에 재사용)
#   ^ ... $        : 줄 '전체'가 이 모양일 때만 일치 (발언 중간의 '2:30'은 걸리지 않음)
#   \d{1,2}:\d{2}  : 숫자 1~2개, 콜론, 숫자 2개 (00:07)
#   (:\d{2})?      : ':숫자 2개'가 있어도 없어도 됨 (1시간 넘는 녹음의 01:02:15)
TIMESTAMP = re.compile(r"^\d{1,2}:\d{2}(:\d{2})?$")


# 한 줄을 받아 타임스탬프 줄이면 True, 아니면 False를 돌려주는 함수
def is_timestamp(line: str) -> bool:
    # line.strip() = 앞뒤 공백 제거 / .match = 줄 처음부터 패턴과 비교 / bool() = 결과를 True·False로 바꾼다
    return bool(TIMESTAMP.match(line.strip()))


# ── 2. 정규식 탐지 규칙 ──────────────────────
# 음성 변환 앱이 숫자를 한글로 받아 적을 때 쓰는 글자들 (공=0, 영=0, 일=1 ... 구=9)
KOR_DIGITS = "공영일이삼사오육칠팔구"

# (유형 이름, 패턴) 목록. 나열 순서 = 우선순위 (앞일수록 겹칠 때 라벨을 차지한다)
# 정규식 공통 기호:
#   (?<!\d) : '바로 앞 글자가 숫자가 아닐 것' — 더 긴 숫자의 중간부터 잘못 잡지 않게
#   (?!\d)  : '바로 뒤 글자가 숫자가 아닐 것' — 더 긴 숫자의 앞부분만 잘못 잡지 않게
#   \d{n}   : 숫자 정확히 n개 / \d{a,b} : 숫자 a~b개 / [-\s]? : 하이픈이나 공백이 있어도 없어도 됨
REGEX_RULES = [
    # 주민등록번호: 6자리 - 7자리. 뒷자리 첫 글자는 성별·세기 숫자라 1~8만 허용
    ("주민등록번호", re.compile(r"(?<!\d)\d{6}\s?-\s?[1-8]\d{6}(?!\d)")),
    # 카드번호: 4자리씩 네 묶음, 사이에 하이픈이나 공백
    ("카드번호", re.compile(r"(?<!\d)\d{4}[-\s]\d{4}[-\s]\d{4}[-\s]\d{4}(?!\d)")),
    # 전화번호: 0으로 시작(010, 02 등) + 3~4자리 + 4자리, 구분자는 하이픈·공백·점 또는 없음
    ("전화번호", re.compile(r"(?<!\d)0\d{1,2}[-\s.]?\d{3,4}[-\s.]?\d{4}(?!\d)")),
    # 계좌번호: 숫자 묶음 세 개를 하이픈으로 연결. 마지막 묶음을 4자리 이상으로 둬서 2026-11-05 같은 날짜는 안 걸리게
    #   (주의: 010-3847-2915 같은 휴대전화도 이 모양에 맞는다 — D41 실습에서 본 바로 그 오탐.
    #    전화번호 규칙이 더 앞에 있으므로, 둘이 겹치면 merge_spans가 '전화번호' 라벨을 남긴다)
    ("계좌번호", re.compile(r"(?<!\d)\d{3,6}-\d{2,6}-\d{4,8}(?!\d)")),
    # 이메일: 영문·숫자·기호 + @ + 도메인 + . + 영문 2글자 이상 (.co.kr처럼 점이 여러 번이어도 됨)
    ("이메일", re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")),
    # 한글로 받아 적힌 휴대전화: '공일공/영일영' 으로 시작 + 한글 숫자 7~8개 (사이 공백 허용)
    #   rf"..." = 정규식용 글자(r)이면서 {KOR_DIGITS} 자리에 변수 값을 끼워 넣는 글자(f)
    #   {{7,8}} = f-문자열 안에서 중괄호를 글자 그대로 쓰려면 두 번 겹쳐 써야 해서 (결과는 {7,8})
    #   '공일공'으로 시작하는 경우만 잡는 이유: '이삼사' 같은 글자는 일반 단어에도 나와서 오탐이 많아지기 때문
    ("전화번호", re.compile(rf"[공영]일[공영](?:\s*[{KOR_DIGITS}]){{7,8}}")),
    # 기관명: 한글·영문·숫자 1글자 이상 + 기관을 뜻하는 끝말 (한빛고등학교, 한결전자, 한빛학원)
    #   끝말 앞에 반드시 글자가 붙어 있어야 해서, '학원 끝나는'처럼 띄어 쓴 일반 단어는 걸리지 않는다
    ("기관", re.compile(r"[가-힣A-Za-z0-9]+(?:초등학교|중학교|고등학교|대학교|대학원|학원|병원|의원|센터|전자|주식회사)")),
]

# 모델 탐지의 우선순위 값 — 정규식(0~6)보다 항상 뒤로 밀리도록 큰 수로 둔다
MODEL_PRIORITY = 100

# 모델이 붙이는 영어 라벨 → 화면에 보일 한국어 유형 이름 대응표
MODEL_LABEL_MAP = {
    "private_person": "이름",
    "private_address": "주소",
    "private_phone": "전화번호",
    "private_email": "이메일",
    "private_date": "날짜",
    "account_number": "계좌번호",
}


# 모델 라벨을 한국어 유형으로 바꾸는 함수
def model_label(raw_label: str) -> str:
    # .get(열쇠, 기본값) = 표에 있으면 그 값을, 없으면 기본값을 돌려준다
    # 표에 없는 라벨도 '기타개인정보'로 가린다 — 모르는 것은 안전한 쪽으로(fail-closed). 표에 없다고 그냥 통과시키면 유출된다
    return MODEL_LABEL_MAP.get(raw_label, "기타개인정보")


# 한 줄에서 정규식 규칙을 전부 돌려 탐지 목록을 돌려주는 함수
def find_regex(line: str) -> list[dict]:
    # 탐지 결과를 담을 빈 리스트
    spans = []
    # enumerate = 목록을 꺼내면서 순번(0, 1, 2...)도 함께 준다 → 그 순번을 우선순위로 쓴다
    for priority, (label, pattern) in enumerate(REGEX_RULES):
        # finditer = 한 줄 안에서 패턴에 맞는 곳을 '모두' 하나씩 찾아 준다 (search는 첫 번째 하나만)
        for m in pattern.finditer(line):
            # 찾은 곳 하나를 사전으로 기록한다. start/end = 원문에서의 시작·끝 위치(끝은 그 글자 '바로 다음' 위치)
            spans.append({
                "start": m.start(),
                "end": m.end(),
                "type": label,
                "source": "regex",
                "priority": priority,
            })
    # 이 줄에서 찾은 전부를 돌려준다
    return spans


# ── 3. 조각 합치기 ───────────────────────────
# 정규식 결과와 모델 결과를 한데 모은 목록을 받아, 조각을 합치고 겹침을 정리한 목록을 돌려주는 함수
def merge_spans(line: str, spans: list[dict]) -> list[dict]:
    # 시작 위치 순으로 정렬한다. 시작이 같으면 우선순위 높은(숫자 작은) 것이 먼저 오게
    #   key=lambda s: (...) = '각 항목에서 이 값을 꺼내 그 값으로 정렬하라'는 짧은 함수
    ordered = sorted(spans, key=lambda s: (s["start"], s["priority"]))
    # 합친 결과를 담을 빈 리스트
    merged = []
    # 정렬된 탐지를 하나씩 꺼낸다
    for s in ordered:
        # 이미 담긴 것이 하나라도 있으면, 바로 앞(마지막)에 담긴 것과 비교한다
        if merged:
            # 마지막에 담긴 탐지
            last = merged[-1]
            # 겹침: 지금 것의 시작이 앞 것의 끝보다 앞이면 두 범위가 포개진 것
            overlap = s["start"] < last["end"]
            # 앞 것의 끝과 지금 것의 시작 사이 글자 (붙어 있으면 빈 글자 "")
            gap = line[last["end"]:s["start"]]
            # 같은 유형 + 겹치지 않음 + 사이가 공백 1칸 이하 → 같은 덩어리의 조각으로 본다 ('박지'+'영', '서울시'+' '+'노원')
            touching_same = (
                s["type"] == last["type"]
                and s["start"] >= last["end"]
                and len(gap) <= 1
                and gap.strip() == ""
            )
            # 둘 중 하나라도 해당하면 새로 담지 않고 앞 것에 합친다
            if overlap or touching_same:
                # 끝 위치를 둘 중 더 뒤쪽으로 넓힌다 — 합집합. 애매하면 더 많이 가리는 쪽(fail-closed)
                last["end"] = max(last["end"], s["end"])
                # 지금 것의 우선순위가 더 높으면(숫자가 작으면) 라벨·출처·우선순위를 지금 것으로 바꾼다
                #   예: 모델이 '계좌번호'라 한 090315-4123456 → 정규식의 '주민등록번호'가 이긴다
                if s["priority"] < last["priority"]:
                    last["type"] = s["type"]
                    last["source"] = s["source"]
                    last["priority"] = s["priority"]
                # 합쳤으니 새로 담지 않고 다음 탐지로 넘어간다
                continue
        # 합칠 대상이 아니면 새 덩어리로 담는다
        #   dict(s) = 복사본을 담는다 — 위에서 last["end"]를 고칠 때 원래 목록의 사전까지 바뀌지 않게
        merged.append(dict(s))
    # 정리된 목록을 돌려준다
    return merged

Writing app/detectors.py


In [7]:
# importlib(이미 불러온 파일을 다시 불러오는 도구)를 가져온다
import importlib
# 방금 만든 detectors 파일을 det라는 짧은 이름으로 가져온다
import app.detectors as det
# 다시 불러온다 — detectors.py를 고친 뒤 이 셀만 다시 실행해도 커널 재시작 없이 새 내용이 반영되게
importlib.reload(det)

# ── 시험 1: 샘플 녹취록 전체에 정규식만 돌려 보기 ──
# 합치기 전·후 개수를 셀 변수
raw_total, total = 0, 0
# 셀 2에서 만든 (타임스탬프, 발언) 목록을 하나씩 꺼낸다
for ts, text in utterances:
    # 정규식 탐지
    found = det.find_regex(text)
    # 합치기 전 개수를 더한다
    raw_total += len(found)
    # 정규식끼리 겹친 것 정리 (010-3847-2915가 전화번호·계좌번호 둘 다로 잡히는 오탐 정리)
    spans = det.merge_spans(text, found)
    # 아무것도 없으면 다음 줄로
    if not spans:
        continue
    # 타임스탬프 출력
    print(f"[{ts}]")
    # 찾은 것을 하나씩 출력 (원문에서 위치로 잘라 보여 준다)
    for sp in spans:
        total += 1
        print(f"   {sp['type']:<8} {text[sp['start']:sp['end']]!r}")
# 합치기 전·후 개수 출력
print(f"정규식 탐지: 합치기 전 {raw_total}개 → 합친 뒤 {total}개")

# ── 시험 2: 셀 3에서 본 '모델 조각'을 손으로 넣어 합치기 확인 ──
print("\n--- 합치기 시험 ---")
# 셀 3 출력에서 1순위 모델이 실제로 돌려준 조각 위치를 그대로 옮겨 적은 것
line = "그리고 학교에서 공문 처리하려면 주민번호가 필요하다고 해서요. 090315-4123456 맞나요?"
# 모델이 '계좌번호(account_number)'라고 잘못 붙인 두 조각
model_parts = [
    {"start": 35, "end": 47, "type": "계좌번호", "source": "model", "priority": det.MODEL_PRIORITY},
    {"start": 47, "end": 49, "type": "계좌번호", "source": "model", "priority": det.MODEL_PRIORITY},
]
# 정규식 결과 + 모델 조각을 함께 넣어 합친다
merged = det.merge_spans(line, det.find_regex(line) + model_parts)
# (원문 조각, 유형, 출처) 형태로 출력한다
print([(line[m["start"]:m["end"]], m["type"], m["source"]) for m in merged])

# 이름 조각 '박지'+'영' 합치기
line2 = "네 안녕하세요. 오늘 상담 맡게 된 진로상담사 박지영입니다. 편하게 앉으세요."
name_parts = [
    {"start": 26, "end": 28, "type": "이름", "source": "model", "priority": det.MODEL_PRIORITY},
    {"start": 28, "end": 29, "type": "이름", "source": "model", "priority": det.MODEL_PRIORITY},
]
merged2 = det.merge_spans(line2, name_parts)
print([(line2[m["start"]:m["end"]], m["type"], m["source"]) for m in merged2])

[00:25]
   기관       '한빛고등학교'
[00:48]
   전화번호     '010-3847-2915'
[00:59]
   전화번호     '공일공 칠이육삼 사팔일오'
[01:10]
   이메일      'seoyeon.kim@example.com'
   이메일      'jh.lee77@example.co.kr'
[01:31]
   주민등록번호   '090315-4123456'
[01:44]
   계좌번호     '123401-04-567890'
[02:40]
   기관       '한결전자'
[03:14]
   기관       '한빛학원'
[03:24]
   전화번호     '02-123-4567'
정규식 탐지: 합치기 전 11개 → 합친 뒤 10개

--- 합치기 시험 ---
[('090315-4123456', '주민등록번호', 'regex')]
[('박지영', '이름', 'model')]


In [8]:
%%writefile app/model_service.py
# 위 줄(%%writefile)은 이 셀의 내용을 실행하지 않고 app/model_service.py 라는 파일로 저장하라는 주피터 명령이다
"""
모델 부품 + 전체 조립: 모델 로드, 모델 탐지, 녹취록 마스킹(가명 처리 포함).
"""
# os(운영체제와 대화하는 모듈)를 가져온다 — 환경 변수에서 모델 이름을 읽으려고
import os
# time(시간 모듈)을 가져온다 — 처리 시간을 재려고
import time
# Counter(개수를 세는 사전)를 가져온다 — 유형별 개수와 가명 순번을 세려고
from collections import Counter

# torch를 가져온다 — GPU 사용 가능 여부를 묻기 위해
import torch
# pipeline을 가져온다 — 토크나이저+모델+후처리를 한 번에 묶어 주는 도구
from transformers import pipeline

# 4-A에서 만든 규칙 부품들을 가져온다
from app.detectors import (
    MODEL_PRIORITY,   # 모델 탐지의 우선순위 값 (100)
    find_regex,       # 정규식 탐지
    is_timestamp,     # 타임스탬프 줄 판별
    merge_spans,      # 조각 합치기·겹침 정리
    model_label,      # 모델 영어 라벨 → 한국어 유형
)

# 모델 이름(또는 로컬 경로)을 환경 변수 PII_MODEL에서 읽고, 없으면 기본값을 쓴다
#   나중에 오프라인 실행으로 바꿀 때 코드는 그대로 두고 이 환경 변수만 로컬 경로로 바꾸면 된다
MODEL_NAME = os.environ.get("PII_MODEL", "FrameByFrame/korean-pii-e5-base")
# 한 번에 모델에 넣을 줄 수
BATCH_SIZE = 16


# 모델을 불러와 돌려주는 함수 — 서버가 켜질 때 한 번만 부른다 (요청마다 부르면 매번 수백 MB를 다시 읽는다)
def load_model():
    # GPU가 있으면 0번 GPU를, 없으면 -1(CPU)을 쓰도록 정한다
    device = 0 if torch.cuda.is_available() else -1
    # 토큰 분류 pipeline을 만들어 돌려준다 (Step 2 셀 3에서 검증한 설정 그대로)
    return pipeline(
        "token-classification",          # 태스크: 토큰(단어 조각)마다 라벨을 붙이는 분류
        model=MODEL_NAME,                # 모델 이름 또는 로컬 경로
        device=device,                   # GPU 또는 CPU
        aggregation_strategy="simple",   # 토큰 라벨을 덩어리로 묶는다 (이 모델은 조각으로 나오지만 merge_spans가 마저 합친다)
    )


# 여러 줄을 한 번에 모델에 넣고, 줄마다 탐지 목록을 돌려주는 함수
#   반환 모양: [[줄1의 탐지들], [줄2의 탐지들], ...] — 넣은 줄 순서와 같다
def find_model(ner, lines: list[str], threshold: float) -> list[list[dict]]:
    # 넣을 줄이 하나도 없으면 모델을 부르지 않고 빈 목록을 돌려준다 (빈 목록을 모델에 넣으면 에러가 날 수 있어서)
    if not lines:
        return []
    # 줄 목록을 통째로 넣는다 → 모델이 BATCH_SIZE개씩 묶어 처리한다 (한 줄씩 부르는 것보다 GPU를 덜 놀린다)
    raw_results = ner(lines, batch_size=BATCH_SIZE)
    # 줄마다의 결과를 담을 빈 리스트
    all_spans = []
    # 줄 하나의 결과(raw)를 하나씩 꺼낸다
    for raw in raw_results:
        # 이 줄의 탐지를 담을 빈 리스트
        spans = []
        # 탐지 하나(r)를 하나씩 꺼낸다
        for r in raw:
            # 확신도가 기준보다 낮으면 버린다
            if r["score"] < threshold:
                continue
            # 4-A의 정규식 결과와 똑같은 모양의 사전으로 바꾼다 — 같은 모양이어야 merge_spans에 한데 넣을 수 있다
            spans.append({
                "start": int(r["start"]),            # int() = 모델이 numpy 숫자로 줄 수 있어서 일반 정수로 바꾼다
                "end": int(r["end"]),
                "type": model_label(r["entity_group"]),  # 영어 라벨 → 한국어 유형 (모르는 라벨은 '기타개인정보')
                "source": "model",
                "priority": MODEL_PRIORITY,
            })
        # 이 줄의 결과를 전체 목록에 넣는다
        all_spans.append(spans)
    # 줄 순서대로 정리된 결과를 돌려준다
    return all_spans


# 순번(0, 1, 2...)을 가명 글자(A, B, C...)로 바꾸는 작은 도우미 함수
def _letters(n: int) -> str:
    # 0~25는 A~Z, 26부터는 27, 28처럼 숫자로 (chr = 번호 → 글자, ord("A") = 'A'의 번호 65)
    return chr(ord("A") + n) if n < 26 else str(n + 1)


# 녹취록 전체를 받아 가린 결과와 요약을 돌려주는 함수 — 서버의 /predict가 부를 핵심 함수
def mask_transcript(ner, text: str, threshold: float = 0.5,
                    pseudonymize: bool = True) -> dict:
    # 처리 시작 시각을 잰다 (perf_counter = 짧은 시간 측정에 쓰는 정밀 시계)
    t0 = time.perf_counter()
    # 줄 단위로 나눈다. splitlines() 대신 split("\n")을 쓰는 이유: 마지막 빈 줄까지 정확히 보존해서
    #   나중에 "\n".join()으로 다시 합쳤을 때 원본과 형식이 완전히 같게 하려고
    lines = text.split("\n")

    # 발언 줄의 '줄 번호'와 '타임스탬프'를 따로 모은다
    speech_idx, speech_ts = [], []
    # 바로 앞에서 본 타임스탬프 (처음엔 없음)
    current_ts = ""
    # enumerate = 줄 번호(i)와 줄 내용(line)을 함께 꺼낸다
    for i, line in enumerate(lines):
        # 빈 줄은 건너뛴다 (그대로 남겨 둔다)
        if not line.strip():
            continue
        # 타임스탬프 줄이면 기억만 하고 건너뛴다 (가리지 않고 그대로 남겨 둔다)
        if is_timestamp(line):
            current_ts = line.strip()
            continue
        # 여기까지 왔으면 발언 줄 — 줄 번호와 그 줄의 타임스탬프를 기록한다
        speech_idx.append(i)
        speech_ts.append(current_ts)

    # 발언 줄만 골라 한 번에 모델에 넣는다 — [lines[i] for i in speech_idx] = 발언 줄 번호에 해당하는 줄들의 목록
    model_spans = find_model(ner, [lines[i] for i in speech_idx], threshold)

    # 가명 대응표: (유형, 원문) → 가명. 함수 안에서만 쓰고 응답·로그에 내보내지 않는다 (원본 복원 열쇠라서)
    pseudonyms = {}
    # 유형별로 가명을 몇 개 만들었는지 센다 (다음 가명 글자를 정하려고)
    type_counters = Counter()
    # 유형별 탐지 개수 (응답의 summary)
    summary = Counter()
    # 탐지 목록 (응답의 entities — 원문 글자는 넣지 않는다)
    entities = []

    # k = 발언 줄 중 몇 번째인지, i = 전체 줄 중 몇 번째인지
    for k, i in enumerate(speech_idx):
        # 이번 발언 줄
        line = lines[i]
        # 정규식 결과 + 모델 결과를 합쳐 조각·겹침을 정리한다
        spans = merge_spans(line, find_regex(line) + model_spans[k])

        # 탐지 하나씩: 가명(tag)을 정하고, 요약·목록에 기록한다
        for sp in spans:
            # 가릴 원문 조각 (대응표의 열쇠로만 쓰고 밖으로 내보내지 않는다)
            original = line[sp["start"]:sp["end"]]
            # 가명 처리를 켰으면
            if pseudonymize:
                # 같은 유형 + 같은 글자면 같은 사람으로 보고 같은 가명을 쓴다
                key = (sp["type"], original)
                # 처음 나온 것이면 새 가명을 만든다
                if key not in pseudonyms:
                    # 예: 이름 유형의 첫 번째 → [이름_A], 두 번째 → [이름_B]
                    pseudonyms[key] = f"[{sp['type']}_{_letters(type_counters[sp['type']])}]"
                    # 이 유형의 가명 순번을 하나 올린다
                    type_counters[sp["type"]] += 1
                # 이 탐지에 붙일 가명
                sp["tag"] = pseudonyms[key]
            # 가명 처리를 껐으면 유형 이름만으로 가린다
            else:
                sp["tag"] = f"[{sp['type']}]"
            # 유형별 개수를 하나 올린다
            summary[sp["type"]] += 1
            # 목록에 유형·타임스탬프·출처만 기록한다 (원문 글자는 넣지 않음 → 응답 전체에 개인정보가 없다)
            entities.append({"type": sp["type"], "timestamp": speech_ts[k], "source": sp["source"]})

        # 뒤에서부터(start가 큰 것부터) 바꿔 끼운다
        #   앞에서부터 바꾸면 '김서연'(3글자) → '[이름_D]'(6글자)처럼 길이가 달라져,
        #   그 뒤에 있는 탐지들의 start·end 위치가 전부 밀려 엉뚱한 곳을 가리키게 되기 때문
        for sp in sorted(spans, key=lambda s: s["start"], reverse=True):
            # 앞부분 + 가명 + 뒷부분으로 줄을 다시 만든다 (문자열은 직접 고칠 수 없어 새로 만들어 끼운다)
            line = line[:sp["start"]] + sp["tag"] + line[sp["end"]:]
        # 가린 줄을 원래 자리에 넣는다
        lines[i] = line

    # 결과를 사전으로 돌려준다 (schemas.py의 MaskResponse에서 success만 빠진 모양)
    return {
        "masked_text": "\n".join(lines),                        # 줄들을 다시 하나로 — 타임스탬프·빈 줄도 원래 자리 그대로
        "summary": dict(summary),                               # Counter → 일반 사전 (JSON으로 바꿀 수 있게)
        "entities": entities,
        "processing_ms": round((time.perf_counter() - t0) * 1000, 1),  # 걸린 초 × 1000 = 밀리초, 소수 첫째 자리까지
    }

Writing app/model_service.py


In [9]:
# 이미 불러온 파일을 다시 불러오는 도구
import importlib
# 정규식(re)을 가져온다 — 타임스탬프 줄 개수를 세려고
import re
# 파일 경로 도구
from pathlib import Path
# 4-A, 4-B 파일을 가져온다
import app.detectors, app.model_service
# 두 파일을 다시 불러온다 — 고친 뒤 이 셀만 다시 실행해도 반영되게 (detectors를 먼저: model_service가 detectors를 쓰므로)
importlib.reload(app.detectors)
importlib.reload(app.model_service)
# 다시 불러온 모듈에서 함수 두 개를 꺼낸다
from app.model_service import load_model, mask_transcript

# 모델을 불러온다 (서버에서는 켜질 때 한 번 하는 일)
ner = load_model()
# 샘플 녹취록 전체를 한 덩어리 글자로 읽는다
text = Path("samples/sample_transcript.txt").read_text(encoding="utf-8")
# 마스킹한다 (기본값: threshold 0.5, 가명 처리 켜짐)
result = mask_transcript(ner, text)

# 가린 결과의 앞부분 900글자만 출력한다 (전체는 길어서)
print(result["masked_text"][:900])
# 유형별 개수와 처리 시간을 출력한다
print("\n요약 :", result["summary"])
print("처리 시간 :", result["processing_ms"], "ms")

# ── 자동 역검수 1: 반드시 가려야 할 것이 결과에 남아 있는가 (정답지 A표 기준) ──
must_hide = ["박지영", "이정훈", "김서연", "서연", "한빛고등학교", "2009년 3월 15일",
             "010-3847-2915", "공일공 칠이육삼 사팔일오", "seoyeon.kim@example.com",
             "jh.lee77@example.co.kr", "노원구", "한빛아파트", "상계동", "090315-4123456",
             "123401-04-567890", "한결전자", "최민호", "김서현", "대치동", "한빛학원"]
# 결과 글자 안에 그대로 남아 있는 것만 골라낸다 (비어 있어야 정상)
print("\n남은 개인정보 :", [w for w in must_hide if w in result["masked_text"]])

# ── 자동 역검수 2: 가리면 안 되는 것이 사라졌는가 (정답지 B표 기준) ──
keep = ["2학년", "3반", "95점", "20년", "다음 주 화요일 오후 2시", "4시", "진로상담사", "간호사", "생명과학"]
# 결과 글자에서 사라진 것만 골라낸다 (비어 있어야 정상)
print("사라진 일반 단어 :", [w for w in keep if w not in result["masked_text"]])

# ── 자동 역검수 3: 타임스탬프 줄이 그대로 보존됐는가 ──
#   (?m) = 여러 줄 모드: ^와 $가 '각 줄'의 시작·끝을 뜻하게 한다
pattern = r"(?m)^\d\d:\d\d$"
print("타임스탬프 줄 :", len(re.findall(pattern, result["masked_text"])), "/", len(re.findall(pattern, text)))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1860.35it/s]



00:00
네 안녕하세요. 오늘 상담 맡게 된 진로상담사 [이름_A]입니다. 편하게 앉으세요. 

00:07
안녕하세요. 저는 [이름_B]이 아빠 [이름_C]이고요 얘가 [이름_D]이에요. 아 성이 달라서 헷갈리실 수 있는데 재혼 가정이라서요. 

00:21
네 괜찮습니다. [이름_B] 학생은 지금 몇 학년이에요? 

00:25
[기관_A] 2학년이요. 3반이에요. 

00:31
네. 상담 전에 서류에 적을 게 있어서 몇 가지만 여쭤볼게요. 생년월일이 어떻게 되죠? 

00:38
[날짜_A]이요. 

00:43
연락 받으실 번호는 어머님 아니 아버님 번호로 할까요? 

00:48
네 제 번호로 해 주세요. [전화번호_A]입니다. 

00:55
학생 본인 번호도 하나 받아 둘게요. 

00:59
[전화번호_B]요. 

01:05
이메일로 결과지를 보내 드리는데 어디로 보내 드릴까요? 

01:10
[이메일_A]으로 보내 주세요. 아니면 제 거 [이메일_B]도 괜찮아요. 

01:19
네. 주소는 [주소_A] 맞으시죠? 

01:27
네 맞아요. [주소_B]이요. 

01:31
그리고 학교에서 공문 처리하려면 주민번호가 필요하다고 해서요. [주민등록번호_A] 맞나요? 

01:40
네 맞습니다. 

01:44
상담비 환불 계좌는 [계좌번호_A] 예금주 [이름_C] 이렇게 받아 둘게요. 

01:53
네. 

01:56
자 그럼 이제 본격적으로 시작해 볼게요. [이름_B]이는 요즘 어떤 것에 관심이 있어요? 

02:03
저는 사람들 얘기 들어주는 게 좋아요. 친구들이 고민 있으면 저한테 많이 와요. 

02:11
그렇구나. 그럴 때 기분이 어때요? 

02:14
뭔가 제가 쓸모 있는 사람이 된 것 같아서 좋아요. 근데 성적이 별로라서 걱정돼요. 

02:22
성적 얘기는 잠깐 뒤에 하고

요약 : {'이름': 13, '기관': 3, '날짜': 1, '전화번호': 3, '이메일': 2, '주소': 3, '주민등록번호': 1, '계좌번호': 1}
처리 시간 : 95

In [10]:
%%writefile app/auth.py
# 위 줄(%%writefile)은 이 셀의 내용을 실행하지 않고 app/auth.py 라는 파일로 저장하라는 주피터 명령이다
"""
API Key 인증 모듈 (D52 Own Project 강의안 3.2절 그대로 재사용).
"""
# HTTPException(에러 응답을 만드는 도구), Header(요청 헤더에서 값을 꺼내는 표시)를 가져온다
from fastapi import HTTPException, Header

# 허용된 키 → 사용자 이름 대응표 (강의안의 시험용 키. 실제 운영이라면 코드 밖 환경 변수·해시 목록으로 — README 보완점)
VALID_API_KEYS = {
    "test-key-001": "사용자A",
    "test-key-002": "사용자B",
}


# 엔드포인트보다 먼저 실행되는 인증 함수 (Depends로 연결된다)
#   x_api_key: 헤더 'X-API-Key'의 값. FastAPI가 밑줄(_)을 하이픈(-)으로 바꿔 찾아 준다. 없으면 None
async def verify_api_key(x_api_key: str = Header(None)) -> str:
    # 헤더가 아예 없으면 401 (누군지 모름 = 인증 실패)
    if x_api_key is None:
        raise HTTPException(
            status_code=401,
            detail="API Key가 필요합니다. X-API-Key 헤더를 포함해 주세요.",
        )
    # 헤더는 있는데 표에 없는 키면 401
    if x_api_key not in VALID_API_KEYS:
        raise HTTPException(
            status_code=401,
            detail="유효하지 않은 API Key입니다.",
        )
    # 통과하면 사용자 이름을 돌려준다 → 엔드포인트의 user 매개변수로 들어간다 (로그에 '누가' 썼는지 남기려고)
    return VALID_API_KEYS[x_api_key]

Writing app/auth.py


In [11]:
%%writefile app/main.py
# 위 줄(%%writefile)은 이 셀의 내용을 실행하지 않고 app/main.py 라는 파일로 저장하라는 주피터 명령이다
"""
녹취록 개인정보 마스킹 API 서버.
흐름: 인증(401) → 입력 검증(422/400/413) → 모델 준비 확인(503) → 전용 스레드에서 마스킹(500) → 응답(200)
"""
# asyncio(이벤트 루프 도구)를 가져온다 — run_in_executor, create_task를 쓰려고
import asyncio
# logging(기록 도구)을 가져온다 — 처리 기록을 남기려고 (원문은 절대 남기지 않음)
import logging
# uuid(겹치지 않는 번호 생성기)를 가져온다 — 요청마다 추적 번호(request_id)를 붙이려고
import uuid
# ThreadPoolExecutor(일꾼 스레드 대기조)를 가져온다 — 오래 걸리는 모델 작업을 이벤트 루프 밖 스레드에 맡기려고
from concurrent.futures import ThreadPoolExecutor
# asynccontextmanager(비동기 '준비-정리' 틀을 만드는 표시)를 가져온다 — lifespan을 만들려고
from contextlib import asynccontextmanager

# FastAPI의 부품들을 가져온다
#   Depends(먼저 실행할 함수 연결), File·Form·UploadFile(파일 업로드 받기), HTTPException(에러 응답), Request(요청 정보)
from fastapi import Depends, FastAPI, File, Form, HTTPException, Request, UploadFile
# 입력 검증 실패 에러의 종류를 가져온다 — 이 에러가 났을 때 우리가 직접 응답을 만들려고
from fastapi.exceptions import RequestValidationError
# JSON 응답을 직접 만드는 도구
from fastapi.responses import JSONResponse
# Pydantic의 검증 실패 에러 — 파일 내용을 MaskRequest로 검사할 때 잡으려고
from pydantic import ValidationError

# 우리가 만든 부품들을 가져온다
from app.auth import verify_api_key                               # 5A-1: API Key 인증
from app.model_service import MODEL_NAME, load_model, mask_transcript  # 4-B: 모델 로드와 마스킹
from app.schemas import MaskRequest, MaskResponse                 # 3: 입력·출력 설계도

# 로그 기본 설정: INFO 등급 이상을 '시각 등급 메시지' 모양으로 화면에 찍는다
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
# 이 서버의 로거 (이름이 같으면 프로그램 어디서든 같은 로거)
logger = logging.getLogger("pii_api")

# 업로드 파일 최대 크기 (200KB)
MAX_FILE_BYTES = 200_000
# 모델 전용 일꾼 대기조. GPU 모델은 한 번에 하나씩 → 일꾼 1명
executor = ThreadPoolExecutor(max_workers=1)
# 서버 상태 보관함: 모델 객체, 상태 글자(loading/healthy/error), 로딩 작업 손잡이
state = {"ner": None, "status": "loading", "task": None}


# 모델을 뒤에서 불러오는 함수 — 서버는 바로 요청을 받기 시작하고, 그동안 /health는 loading을 답한다
async def _load_model_in_background():
    # 지금 돌고 있는 이벤트 루프를 가져온다
    loop = asyncio.get_running_loop()
    # 로드가 실패해도 서버 전체가 죽지 않도록 감싼다
    try:
        # load_model은 오래 걸리는 동기 함수 → 일꾼 스레드에 맡기고 await로 양보 (그동안 /health가 답할 수 있다)
        #   load_model 뒤에 괄호를 붙이지 않는다 — 붙이면 이 자리(루프 스레드)에서 바로 실행돼 루프가 막힌다
        state["ner"] = await loop.run_in_executor(executor, load_model)
        # 성공하면 상태를 healthy로
        state["status"] = "healthy"
        # 성공 기록 (모델 이름만)
        logger.info("모델 로드 완료: %s", MODEL_NAME)
    # 실패하면
    except Exception as e:
        # 상태를 error로 → 이후 /predict는 503
        state["status"] = "error"
        # 에러 종류 이름만 기록한다
        logger.error("모델 로드 실패: %s", type(e).__name__)


# 서버가 켜질 때와 꺼질 때 할 일 (yield 앞 = 켜질 때, 뒤 = 꺼질 때)
@asynccontextmanager
async def lifespan(app: FastAPI):
    # 모델 로딩을 '뒤에서 도는 작업'으로 띄운다. 손잡이를 state에 보관 — 보관하지 않으면 도중에 정리(삭제)될 수 있어서
    state["task"] = asyncio.create_task(_load_model_in_background())
    # 여기서 서버가 요청을 받기 시작한다
    yield
    # 서버가 꺼질 때 일꾼 대기조를 정리한다
    executor.shutdown(wait=False)


# FastAPI 앱을 만든다. title·description·version은 Swagger 화면 맨 위에 보인다
app = FastAPI(
    title="녹취록 개인정보 마스킹 API",
    description="갤럭시 녹음 앱에서 내보낸 녹취록의 개인정보를 가명으로 바꿔 돌려줍니다.",
    version="1.0.0",
    lifespan=lifespan,
)


# 검증 에러 목록에서 '위치(loc)·이유(msg)·종류(type)'만 남기는 함수 — 입력 원문(input)은 뺀다 (개인정보라서)
def _safe_errors(errors) -> list[dict]:
    # 에러 하나(e)마다 세 가지만 골라 새 사전을 만든다. .get(열쇠, 기본값) = 없으면 기본값
    return [
        {"loc": list(e.get("loc", [])), "msg": e.get("msg", ""), "type": e.get("type", "")}
        for e in errors
    ]


# 입력 검증 실패(422)가 나면 FastAPI 기본 처리 대신 이 함수가 응답을 만든다
@app.exception_handler(RequestValidationError)
async def validation_handler(request: Request, exc: RequestValidationError):
    # 추적 번호를 만든다 (uuid 앞 8글자)
    request_id = uuid.uuid4().hex[:8]
    # 원문을 뺀 에러 목록
    safe = _safe_errors(exc.errors())
    # 로그에는 위치와 종류만 남긴다
    logger.warning("입력 검증 실패 request_id=%s errors=%s", request_id,
                   [(e["loc"], e["type"]) for e in safe])
    # 손님에게는 원문 없는 에러 목록 + 추적 번호를 돌려준다
    return JSONResponse(status_code=422, content={"detail": safe, "request_id": request_id})


# /predict와 /predict/file이 함께 쓰는 공통 처리 함수 (같은 코드를 두 번 쓰지 않으려고)
async def _run_mask(req: MaskRequest, user: str) -> MaskResponse:
    # 모델이 아직 로딩 중이면 503 (서버 탓이지만 시간이 지나면 해결됨 → 다시 시도하라고 안내)
    if state["status"] == "loading":
        raise HTTPException(status_code=503, detail="모델을 불러오는 중입니다. 잠시 후 다시 시도해 주세요.")
    # 로딩에 실패한 상태면 503
    if state["status"] != "healthy":
        raise HTTPException(status_code=503, detail="모델을 사용할 수 없습니다. 관리자에게 문의해 주세요.")
    # 이 요청의 추적 번호
    request_id = uuid.uuid4().hex[:8]
    # 지금 이벤트 루프
    loop = asyncio.get_running_loop()
    # 마스킹 중 에러가 나면 잡아서 500으로 바꾼다
    try:
        # mask_transcript(모델, 글자, 기준값, 가명여부)를 일꾼 스레드에 맡긴다
        #   run_in_executor는 인자를 이름 없이 순서대로만 받는다 → 4-B 함수의 매개변수 순서와 같게 넣는다
        result = await loop.run_in_executor(
            executor, mask_transcript, state["ner"], req.text, req.threshold, req.pseudonymize
        )
    # 어떤 에러든
    except Exception as e:
        # 로그에는 추적 번호·사용자·에러 종류 이름만 남긴다 (에러 메시지에 원문 조각이 섞일 수 있어서)
        logger.error("마스킹 실패 request_id=%s user=%s error=%s", request_id, user, type(e).__name__)
        # 반쯤 가린 결과를 돌려주지 않고 실패로 끝낸다 (fail-closed). 손님에게는 고정 문구 + 추적 번호
        raise HTTPException(
            status_code=500,
            detail=f"마스킹 처리 중 오류가 발생했습니다. (request_id={request_id})",
        )
    # 성공 기록: 원문 대신 글자 수·탐지 건수·처리 시간만
    logger.info("마스킹 완료 request_id=%s user=%s 글자수=%d 탐지=%d건 %.1fms",
                request_id, user, len(req.text), len(result["entities"]), result["processing_ms"])
    # 결과 사전을 펼쳐(**) 응답 설계도에 담는다 + success=True
    return MaskResponse(success=True, **result)


# 상태 확인 — 인증을 걸지 않는다 (모니터링·게이트웨이가 키 없이 물어볼 수 있어야 해서)
@app.get("/health")
async def health():
    # 상태 글자, 모델 이름, 모델이 실제로 올라왔는지
    return {"status": state["status"], "model": MODEL_NAME, "model_loaded": state["ner"] is not None}


# 텍스트를 JSON으로 받아 마스킹 (강의안 필수 엔드포인트)
#   req: MaskRequest → 본문 JSON을 설계도로 검사 (실패하면 422, 이 함수는 실행되지 않음)
#   user: Depends(verify_api_key) → 이 함수보다 인증이 먼저 실행 (실패하면 401)
@app.post("/predict", response_model=MaskResponse)
async def predict(req: MaskRequest, user: str = Depends(verify_api_key)):
    # 공통 처리 함수에 넘긴다
    return await _run_mask(req, user)


# 갤럭시 녹음 앱의 .txt 파일을 받아 마스킹
#   파일과 옵션을 함께 보내야 해서 JSON 대신 multipart 형식 → 옵션은 Form으로 받는다
@app.post("/predict/file", response_model=MaskResponse)
async def predict_file(
    file: UploadFile = File(..., description="갤럭시 녹음 앱에서 내보낸 .txt 파일"),
    threshold: float = Form(0.5, ge=0.0, le=1.0),
    pseudonymize: bool = Form(True),
    user: str = Depends(verify_api_key),
):
    # ① 확장자 검사 (가장 싼 검사라 먼저). file.filename이 None일 수 있어 or ""로 대비, .lower()로 대소문자 무시
    if not (file.filename or "").lower().endswith(".txt"):
        raise HTTPException(status_code=400, detail=".txt 파일만 업로드할 수 있습니다.")
    # ② 크기 검사: 최대 크기보다 '1바이트만 더' 읽는다
    #   다 읽고 나서 재면 거대한 파일이 이미 메모리를 다 차지한 뒤라 보호가 안 된다 (D49 실습의 빈틈)
    data = await file.read(MAX_FILE_BYTES + 1)
    # 1바이트 더 읽혔다면 최대 크기를 넘은 것 → 413 (본문이 너무 큼)
    if len(data) > MAX_FILE_BYTES:
        raise HTTPException(status_code=413, detail=f"파일이 너무 큽니다. 최대 {MAX_FILE_BYTES // 1000}KB까지 가능합니다.")
    # ③ 실제로 UTF-8 글자로 풀어 본다 — 확장자는 손님이 붙인 꼬리표일 뿐, 진짜 판정은 열어 보는 것
    try:
        # utf-8-sig = 파일 맨 앞에 숨은 표식(BOM)이 있으면 떼고 읽는다 (윈도우 메모장 저장 파일 대비)
        text = data.decode("utf-8-sig")
    # UTF-8로 풀리지 않으면 (예: CP949로 저장된 파일, 이미지를 .txt로 이름만 바꾼 파일)
    except UnicodeDecodeError:
        raise HTTPException(status_code=400, detail="UTF-8 텍스트 파일이 아닙니다. 갤럭시 녹음 앱에서 내보낸 .txt 파일인지 확인해 주세요.")
    # ④ /predict와 같은 규칙(길이·공백)으로 검사한다 — 파일 경로는 FastAPI 자동 검사를 거치지 않으므로 직접
    try:
        req = MaskRequest(text=text, threshold=threshold, pseudonymize=pseudonymize)
    # 규칙 위반이면 422, 원문은 빼고
    except ValidationError as e:
        raise HTTPException(status_code=422, detail=_safe_errors(e.errors()))
    # 공통 처리 함수에 넘긴다
    return await _run_mask(req, user)

Writing app/main.py


In [12]:
# 이미 불러온 파일을 다시 불러오는 도구
import importlib
# 서버 파일을 가져온다 (이때 auth.py, model_service.py, schemas.py도 함께 불러와진다)
import app.main
# 다시 불러온다 — main.py를 고친 뒤 이 셀만 다시 실행해도 반영되게
importlib.reload(app.main)

# 서버에 등록된 주소(경로)와 방식(GET/POST)을 하나씩 출력한다 — 우리가 만든 3개가 있는지 확인
for route in app.main.app.routes:
    # route.methods = 그 주소가 받는 방식들(set). getattr(..., None) = 없으면 None (문서용 경로 등)
    methods = getattr(route, "methods", None)
    # 우리가 만든 경로만 골라 출력한다 (/docs, /openapi.json 같은 자동 경로는 제외)
    if route.path in ("/health", "/predict", "/predict/file"):
        print(sorted(methods), route.path)

['GET'] /health
['POST'] /predict
['POST'] /predict/file


In [13]:
# requests(HTTP 요청 도구)를 가져온다 — 노트북에서 서버로 요청을 보내려고
import requests
# 파일 경로 도구
from pathlib import Path

# 서버 주소 — localhost가 아니라 127.0.0.1 (윈도우에서 localhost는 요청마다 약 2초 지연)
API = "http://127.0.0.1:8000"
# 정상 키를 담은 헤더
OK = {"X-API-Key": "test-key-001"}
# 샘플 녹취록 원문(글자)과 원본 바이트
sample_text = Path("samples/sample_transcript.txt").read_text(encoding="utf-8")
sample_bytes = Path("samples/sample_transcript.txt").read_bytes()
# 응답에 섞여 나오면 안 되는 가상 개인정보 문장
LEAK_PROBE = "김민준 010-1234-5678 "

# (시험 이름, 기대 코드, 요청을 보내는 함수) 목록
#   lambda: ... = '나중에 부르면 이 요청을 보내라'는 짧은 함수. 표를 먼저 다 적어 두고 아래에서 하나씩 실행하려고
tests = [
    ("① 헬스체크",            200, lambda: requests.get(f"{API}/health", timeout=10)),
    ("② 정상 텍스트",          200, lambda: requests.post(f"{API}/predict", json={"text": sample_text}, headers=OK, timeout=60)),
    ("③ 키 없음",             401, lambda: requests.post(f"{API}/predict", json={"text": sample_text}, timeout=60)),
    ("④ 틀린 키",             401, lambda: requests.post(f"{API}/predict", json={"text": sample_text}, headers={"X-API-Key": "wrong"}, timeout=60)),
    ("⑤ 공백만",              422, lambda: requests.post(f"{API}/predict", json={"text": "     "}, headers=OK, timeout=60)),
    ("⑥ 5만 자 초과",          422, lambda: requests.post(f"{API}/predict", json={"text": LEAK_PROBE * 3000}, headers=OK, timeout=60)),
    ("⑦ threshold=2",        422, lambda: requests.post(f"{API}/predict", json={"text": "안녕", "threshold": 2}, headers=OK, timeout=60)),
    # files= : 파일 업로드 형식(multipart)으로 보낸다. (파일 이름, 바이트, 종류) 짝
    ("⑧ 파일 업로드",          200, lambda: requests.post(f"{API}/predict/file", files={"file": ("sample_transcript.txt", sample_bytes, "text/plain")}, headers=OK, timeout=60)),
    ("⑨ .md 파일",           400, lambda: requests.post(f"{API}/predict/file", files={"file": ("expected_pii.md", b"# x", "text/markdown")}, headers=OK, timeout=60)),
    # "안녕".encode("cp949") = 윈도우 옛 한글 형식으로 저장된 파일을 흉내 (UTF-8로 풀리지 않음)
    ("⑩ CP949 인코딩 .txt",   400, lambda: requests.post(f"{API}/predict/file", files={"file": ("old.txt", "안녕".encode("cp949"), "text/plain")}, headers=OK, timeout=60)),
]

# 표 머리 출력 (:<20 = 20칸 왼쪽 정렬)
print(f"{'시험':<20}{'기대':>5}{'실제':>6}  판정")
# 통과 개수를 셀 변수
passed = 0
# 시험을 하나씩 꺼내 실행한다
for name, expected, send in tests:
    # 요청을 보낸다 (위에서 적어 둔 lambda를 지금 부른다)
    r = send()
    # 기대 코드와 실제 코드가 같은지
    ok = (r.status_code == expected)
    # 통과면 개수를 올린다
    passed += ok
    # 한 줄 출력
    print(f"{name:<20}{expected:>5}{r.status_code:>6}  {'✅' if ok else '❌'}")

# 요약
print(f"\n통과 {passed} / {len(tests)}")

# ── 추가 확인: 422 응답에 입력 원문이 섞여 나오지 않는가 ──
r = requests.post(f"{API}/predict", json={"text": LEAK_PROBE * 3000}, headers=OK, timeout=60)
# 응답 본문 글자 안에 '김민준'이나 전화번호가 있는지 (없어야 정상)
print("422 응답에 원문 포함 :", ("김민준" in r.text) or ("010-1234-5678" in r.text))
# 응답 본문 그대로 (원문 없이 loc·msg·type·request_id만 있어야 함)
print("422 응답 본문 :", r.json())

# ── 추가 확인: 정상 응답의 요약 ──
print("\n② 정상 응답 요약 :", requests.post(f"{API}/predict", json={"text": sample_text}, headers=OK, timeout=60).json()["summary"])

시험                     기대    실제  판정
① 헬스체크                200   200  ✅
② 정상 텍스트              200   200  ✅
③ 키 없음                401   401  ✅
④ 틀린 키                401   401  ✅
⑤ 공백만                 422   422  ✅
⑥ 5만 자 초과             422   422  ✅
⑦ threshold=2         422   422  ✅
⑧ 파일 업로드              200   200  ✅
⑨ .md 파일              400   400  ✅
⑩ CP949 인코딩 .txt      400   400  ✅

통과 10 / 10
422 응답에 원문 포함 : False
422 응답 본문 : {'detail': [{'loc': ['body', 'text'], 'msg': 'String should have at most 50000 characters', 'type': 'string_too_long'}], 'request_id': 'ae028218'}

② 정상 응답 요약 : {'이름': 13, '기관': 3, '날짜': 1, '전화번호': 3, '이메일': 2, '주소': 3, '주민등록번호': 1, '계좌번호': 1}


In [14]:
%%writefile frontend/app.py
# 위 줄(%%writefile)은 이 셀의 내용을 실행하지 않고 frontend/app.py 라는 파일로 저장하라는 주피터 명령이다
"""
녹취록 개인정보 마스킹 — Streamlit 화면.
실행: streamlit run frontend/app.py --server.port 8501 --server.address 127.0.0.1 --browser.gatherUsageStats false
"""
# hashlib(해시 도구)을 가져온다 — 입력+설정의 '지문'을 만들어, 결과가 지금 입력의 것인지 확인하려고
import hashlib
# os(운영체제와 대화하는 모듈)를 가져온다 — 서버 주소를 환경 변수에서 읽으려고
import os

# requests(HTTP 요청 도구)를 가져온다 — FastAPI 서버로 요청을 보내려고
import requests
# streamlit을 st라는 짧은 이름으로 가져온다
import streamlit as st

# 서버 주소. 환경 변수 PII_API_BASE가 있으면 그 값, 없으면 이 PC의 8000번 (localhost 대신 127.0.0.1 — 2초 지연 방지)
API_BASE = os.environ.get("PII_API_BASE", "http://127.0.0.1:8000")
# 서버 응답을 기다릴 최대 시간(초)
TIMEOUT = 60

# 상태 코드 → 사람이 읽을 안내 문구
ERROR_GUIDE = {
    400: "파일 형식을 확인해 주세요.",
    401: "API Key를 확인해 주세요.",
    413: "파일이 너무 큽니다.",
    422: "입력 내용을 확인해 주세요.",
    500: "서버 처리 중 오류가 났습니다. 잠시 후 다시 시도해 주세요.",
    503: "모델이 아직 준비되지 않았습니다. 잠시 후 다시 시도해 주세요.",
}

# 브라우저 탭 제목·아이콘, 화면을 넓게 쓰기(원문·결과를 좌우로 나란히 보려고)
st.set_page_config(page_title="녹취록 개인정보 마스킹", page_icon="🔒", layout="wide")


# 서버 상태를 물어보는 함수 — 재실행될 때마다 한 번씩 불린다
def check_health():
    # 서버가 꺼져 있어도 화면이 죽지 않게 감싼다
    try:
        # /health에 GET 요청 (5초 안에 답이 없으면 포기)
        r = requests.get(f"{API_BASE}/health", timeout=5)
        # 응답 JSON에서 status 값을 꺼낸다 (없으면 'unknown')
        return r.json().get("status", "unknown")
    # 연결 실패·시간 초과 등 requests의 모든 에러
    except requests.exceptions.RequestException:
        # 서버가 꺼져 있다고 표시
        return "offline"


# 서버에 마스킹을 요청하고, 성공하면 결과 사전을, 실패하면 화면에 안내를 띄우고 None을 돌려주는 함수
def call_api(text, file_name, file_bytes, api_key, threshold, pseudonymize):
    # 키를 입력했으면 X-API-Key 헤더에 담고, 아니면 빈 헤더 (서버가 401을 돌려주게)
    headers = {"X-API-Key": api_key} if api_key else {}
    # 에러를 종류별로 잡기 위해 감싼다
    try:
        # 파일을 올렸으면 /predict/file로
        if file_bytes is not None:
            r = requests.post(
                f"{API_BASE}/predict/file",
                headers=headers,
                # 파일 업로드 형식: (파일 이름, 바이트, 종류)
                files={"file": (file_name, file_bytes, "text/plain")},
                # 옵션은 Form 칸으로 — Form은 글자로만 보낼 수 있어 str()로 바꾼다. True → "true"
                data={"threshold": str(threshold), "pseudonymize": str(pseudonymize).lower()},
                timeout=TIMEOUT,
            )
        # 직접 붙여 넣었으면 /predict로 JSON 요청
        else:
            r = requests.post(
                f"{API_BASE}/predict",
                headers=headers,
                json={"text": text, "threshold": threshold, "pseudonymize": pseudonymize},
                timeout=TIMEOUT,
            )
        # 4xx·5xx면 HTTPError를 일으킨다 — 이 줄이 없으면 requests는 401·422도 성공처럼 넘긴다
        r.raise_for_status()
        # 성공이면 응답 JSON을 사전으로 돌려준다
        return r.json()
    # 서버가 꺼져 있을 때 (except는 위에서부터 첫 일치 하나만 실행 → 구체적인 것부터)
    except requests.exceptions.ConnectionError:
        st.error(f"서버에 연결할 수 없습니다. FastAPI 서버({API_BASE})가 켜져 있는지 확인해 주세요.")
    # 응답이 너무 늦을 때
    except requests.exceptions.Timeout:
        st.error(f"{TIMEOUT}초 안에 응답이 없습니다.")
    # 서버가 4xx·5xx로 답했을 때
    except requests.exceptions.HTTPError:
        # 상태 코드
        code = r.status_code
        # 서버가 보낸 이유(detail)를 꺼내 본다
        try:
            detail = r.json().get("detail")
        # 응답이 JSON이 아니면
        except ValueError:
            detail = None
        # 422처럼 detail이 목록이면 각 항목의 msg만 이어 붙인다
        if isinstance(detail, list):
            detail = " / ".join(d.get("msg", "") for d in detail)
        # 상태 코드 + 안내 문구 + 서버가 보낸 이유를 빨간 상자로 보여 준다 (끝의 공백 두 칸 + \n = 줄바꿈)
        st.error(f"[{code}] {ERROR_GUIDE.get(code, '요청이 실패했습니다.')}  \n{detail or ''}")
    # 실패한 경우 모두 None
    return None


# ── 사이드바 ─────────────────────────────────
# with st.sidebar: 이 블록 안에서 그리는 것은 왼쪽 사이드바에 그려진다
with st.sidebar:
    # 사이드바 제목
    st.header("설정")
    # 서버 상태를 물어본다
    status = check_health()
    # 상태 글자 → 화면 표시
    badge = {"healthy": "🟢 정상", "loading": "🟡 모델 로딩 중", "offline": "🔴 서버 꺼짐"}
    # 표에 없는 상태(예: error)는 빨간 동그라미 + 상태 글자
    st.write(f"서버 상태: {badge.get(status, '🔴 ' + status)}")
    # API Key 입력칸. type="password" = 글자를 ●로 가린다. placeholder = 빈칸일 때 회색 안내 글자
    api_key = st.text_input("API Key", type="password", placeholder="test-key-001")
    # 확신도 기준 슬라이더 (최소, 최대, 기본값, 간격)
    threshold = st.slider("모델 확신도 기준", 0.0, 1.0, 0.5, 0.05,
                          help="낮을수록 더 많이 가립니다(놓침↓, 과잉 마스킹↑)")
    # 가명 처리 체크박스
    pseudonymize = st.checkbox("가명 처리 ([이름_A]처럼 같은 사람은 같은 가명)", value=True)

# ── 본문: 입력 ───────────────────────────────
# 큰 제목
st.title("🔒 녹취록 개인정보 마스킹")
# 제목 아래 작은 설명
st.caption("갤럭시 녹음 앱에서 내보낸 녹취록(.txt)의 이름·연락처·주소 등을 가명으로 바꿉니다. "
           "모든 처리는 이 PC 안의 서버에서만 이루어집니다.")

# 파일 업로드칸 (.txt만 고를 수 있게)
uploaded = st.file_uploader("녹취록 파일 (.txt)", type=["txt"])
# 직접 붙여 넣는 칸. 파일을 올렸으면 비활성화 (둘 중 무엇을 보낼지 헷갈리지 않게)
typed = st.text_area("또는 직접 붙여 넣기", height=150,
                     placeholder="00:00\n안녕하세요 저는 ...", disabled=uploaded is not None)

# 보낼 파일 바이트(없으면 None)와 화면에 보여 줄 원문 글자
file_bytes, text = None, typed
# 파일을 올렸으면
if uploaded is not None:
    # 파일 전체를 바이트로 꺼낸다
    file_bytes = uploaded.getvalue()
    # 원문 표시용으로 UTF-8 글자로 풀어 본다
    try:
        text = file_bytes.decode("utf-8-sig")
    # 풀리지 않으면 표시용 원문은 비워 둔다 (서버로는 그대로 보내 서버가 400으로 판정하게)
    except UnicodeDecodeError:
        text = ""

# 지금 입력+설정의 지문(SHA-256) — 결과가 '지금 입력'의 것인지 확인하는 데 쓴다
input_key = hashlib.sha256(
    (text + f"|{threshold}|{pseudonymize}").encode("utf-8", "ignore") + (file_bytes or b"")
).hexdigest()

# 실행 버튼. 입력이 비어 있으면 누를 수 없게. 누른 그 재실행에서만 True
if st.button("마스킹 실행", type="primary", disabled=not (text.strip() or file_bytes)):
    # 처리 중 빙글빙글 표시
    with st.spinner("개인정보를 찾는 중..."):
        result = call_api(text, uploaded.name if uploaded else None, file_bytes,
                          api_key, threshold, pseudonymize)
    # 성공했으면 결과와 원문·지문·파일 이름을 보관함에 저장 — 다음 재실행(다운로드 클릭 등)에도 남도록
    if result:
        st.session_state["result"] = result
        st.session_state["original"] = text
        st.session_state["result_key"] = input_key
        st.session_state["source_name"] = uploaded.name if uploaded else "pasted.txt"

# ── 본문: 결과 ───────────────────────────────
# 보관함에 결과가 있으면 보여 준다 (버튼을 방금 누르지 않았어도)
if "result" in st.session_state:
    # 보관된 결과
    result = st.session_state["result"]
    # 결과를 만든 입력의 지문이 지금 입력의 지문과 다르면 안내 (이전 결과를 지금 것으로 착각하지 않게)
    if st.session_state["result_key"] != input_key:
        st.info("입력이나 설정이 바뀌었습니다. 아래는 이전 실행 결과입니다. 다시 실행해 주세요.")

    # 결과 소제목
    st.subheader("결과")
    # 유형별 개수
    summary = result["summary"]
    # 숫자 칸 4개를 가로로
    cols = st.columns(4)
    # 가린 총 건수 (사전 값들의 합)
    cols[0].metric("가린 개인정보", f"{sum(summary.values())}건")
    # 유형 수
    cols[1].metric("유형 수", f"{len(summary)}종")
    # 처리 시간 (소수점 없이)
    cols[2].metric("처리 시간", f"{result['processing_ms']:.0f} ms")
    # 정규식이 찾은 것 / 모델이 찾은 것 (True는 1로 세어진다)
    cols[3].metric("정규식 / 모델", "{} / {}".format(
        sum(e["source"] == "regex" for e in result["entities"]),
        sum(e["source"] == "model" for e in result["entities"])))

    # 유형별 개수 가로 막대 그래프
    st.bar_chart(summary, horizontal=True)

    # 좌우 두 칸
    left, right = st.columns(2)
    # 왼쪽: 원문 (읽기 전용)
    left.text_area("원문", st.session_state["original"], height=450, disabled=True)
    # 오른쪽: 마스킹 결과 (복사하기 쉽게 편집 가능 칸으로)
    right.text_area("마스킹 결과", result["masked_text"], height=450)

    # 마스킹 결과를 .txt로 내려받는 버튼 — 누르면 재실행되지만 결과는 보관함에 있어 사라지지 않는다
    st.download_button(
        "마스킹된 텍스트 다운로드",
        data=result["masked_text"].encode("utf-8"),
        file_name="masked_" + st.session_state["source_name"],
        mime="text/plain",
    )

    # 접었다 펴는 상자 안에 탐지 목록 표 (유형·타임스탬프·출처만, 원문 없음)
    with st.expander("탐지 목록 (유형·위치·출처만, 원문 없음)"):
        st.dataframe(result["entities"], width="stretch")

Writing frontend/app.py
